# Lab Assignment - 4
### Ex No.4 - ResNet-152 for Binary Classification of Skin Lesions 
### CS22B1093 Rohan G

-----------------

-------------------

### Importing Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torch
from sklearn.metrics import classification_report, roc_auc_score, f1_score
import torch
import torch.nn as nn
import torchvision.models as models

--------------------

### Data Augmentation

In [26]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

class SkinLesionDataset(Dataset):
    def __init__(self, images_path, csv_path, transform=None):
        self.csv_file = pd.read_csv(csv_path)
        self.csv_file.sort_values(by="image_name", inplace=True)
        self.labels = torch.tensor(self.csv_file["target"].values, dtype=torch.long)
        self.images_path = images_path
        self.transform = transform
        self._make_data()

    def _make_data(self):
        self.data = []
        for i in range(len(self.csv_file)):
            if i % 1000 == 0:
                print(i)
            img_path = os.path.join(self.images_path, self.csv_file.loc[i, "image_name"]) + ".jpg"
            
            if not os.path.exists(img_path):
                print(f"Image not found: {img_path}")
                continue

            try:
                self.data.append((img_path, self.labels[i]))
            except Exception as e:
                print(f"Failed to read image: {img_path}, Error: {e}")
                continue

    def __getitem__(self, index):
        image_path, label = self.data[index]
        image = Image.open(image_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

    def __len__(self):
        return len(self.data)

In [ ]:
train_dataset = SkinLesionDataset("/kaggle/input/dl-lab4-dataset/isic2020/train", "/kaggle/input/dl-lab4-dataset/isic2020/train.csv", transform=transform)
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)

images, labels = next(iter(train_dataloader))

print(images.shape, labels.shape)

0
1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
16000
17000
18000
19000
20000
21000
22000
23000
24000
25000
26000
27000
28000
29000
30000
torch.Size([32, 3, 224, 224]) torch.Size([32])


In [28]:
test_dataset = SkinLesionDataset("/kaggle/input/dl-lab4-dataset/isic2020/test", "/kaggle/input/dl-lab4-dataset/isic2020/test.csv", transform=transform)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=True)

images, labels = next(iter(test_dataloader))
print(images.shape, labels.shape)

0
1000
2000
torch.Size([32, 3, 224, 224]) torch.Size([32])


In [ ]:
class BottleNeck(nn.Module):
    expansion = 4

    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super(BottleNeck, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.conv3 = nn.Conv2d(out_channels, out_channels * self.expansion, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(out_channels * self.expansion)

        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out)
        out = self.bn3(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)

        return out

class ResNet152(nn.Module):
    def __init__(self, num_classes=2):
        super(ResNet152, self).__init__()
        self.in_channels = 64
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        self.layer1 = self._make_layer(BottleNeck, 64, 3)  
        self.layer2 = self._make_layer(BottleNeck, 128, 8, stride=2)  
        self.layer3 = self._make_layer(BottleNeck, 256, 36, stride=2)  
        self.layer4 = self._make_layer(BottleNeck, 512, 3, stride=2) 

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * BottleNeck.expansion, num_classes)

    def _make_layer(self, block, out_channels, num_blocks, stride=1):
        downsample = None
        if stride != 1 or self.in_channels != out_channels * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_channels, out_channels * block.expansion, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels * block.expansion)
            )

        layers = []
        layers.append(block(self.in_channels, out_channels, stride=stride, downsample=downsample))
        self.in_channels = out_channels * block.expansion

        for _ in range(1, num_blocks):
            layers.append(block(self.in_channels, out_channels))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)

        return x

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ResNet152(num_classes=2).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print(model)

ResNet152(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BottleNeck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(

-------------------

### Adam Optimizer and Learning Rate (a)  = 0.001

In [32]:
def train_model(model, train_loader, criterion, optimizer, epochs=2):
    model.train()
    
    for epoch in range(epochs):
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        epoch_loss = running_loss / len(train_loader)
        epoch_acc = 100 * correct / total
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.2f}%")

In [ ]:
train_model(model, train_dataloader, criterion, optimizer, epochs=2)

Epoch [1/2], Loss: 0.1038, Accuracy: 98.20%
Epoch [2/2], Loss: 0.0920, Accuracy: 98.20%


In [ ]:
def test_model(model, test_loader, criterion, device):
    model.eval()  
    correct = 0
    total = 0
    running_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad(): 
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()

            _, predicted = torch.max(outputs, 1)  
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            all_preds.extend(predicted.cpu().numpy()) 
            all_labels.extend(labels.cpu().numpy())  

    avg_loss = running_loss / len(test_loader)
    accuracy = 100 * correct / total

    print(f"Test Loss: {avg_loss:.4f}, Test Accuracy: {accuracy:.2f}%")
    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds))

    try:
        auc_roc = roc_auc_score(all_labels, all_preds)
        print(f"AUC ROC Score: {auc_roc:.4f}")
    except ValueError:
        print("AUC ROC Score: Not applicable for multi-class classification")

    f1 = f1_score(all_labels, all_preds, average='weighted')
    print(f"F1 Score: {f1:.4f}")

test_model(model, test_dataloader, criterion, device)


Test Loss: 0.0975, Test Accuracy: 98.17%

Classification Report:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      2142
           1       0.00      0.00      0.00        40

    accuracy                           0.98      2182
   macro avg       0.49      0.50      0.50      2182
weighted avg       0.96      0.98      0.97      2182

AUC ROC Score: 0.5000
F1 Score: 0.9726


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


-------------------

### Adam Optimizer and Learning Rate (a) =  0.01

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ResNet152(num_classes=2).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [36]:
train_model(model, train_dataloader, criterion, optimizer, epochs=2)

Epoch [1/2], Loss: 0.1304, Accuracy: 98.18%
Epoch [2/2], Loss: 0.0899, Accuracy: 98.21%


In [37]:
test_model(model, test_dataloader, criterion, device)

Test Loss: 0.5496, Test Accuracy: 92.90%

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.94      0.96      2142
           1       0.05      0.17      0.08        40

    accuracy                           0.93      2182
   macro avg       0.52      0.56      0.52      2182
weighted avg       0.97      0.93      0.95      2182

AUC ROC Score: 0.5590
F1 Score: 0.9469


----------------

### SGD Optimizer and Learning Rate (a) =  0.01

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ResNet152(num_classes=2).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

In [39]:
train_model(model, train_dataloader, criterion, optimizer, epochs=2)

Epoch [1/2], Loss: 0.1088, Accuracy: 98.05%
Epoch [2/2], Loss: 0.0920, Accuracy: 98.16%


In [40]:
test_model(model, test_dataloader, criterion, device)

Test Loss: 0.0818, Test Accuracy: 98.17%

Classification Report:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      2142
           1       0.00      0.00      0.00        40

    accuracy                           0.98      2182
   macro avg       0.49      0.50      0.50      2182
weighted avg       0.96      0.98      0.97      2182

AUC ROC Score: 0.5000
F1 Score: 0.9726


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


---------------------

### SGD Optimizer and Learning Rate (a) =  0.001

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ResNet152(num_classes=2).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)

In [42]:
train_model(model, train_dataloader, criterion, optimizer, epochs=2)

Epoch [1/2], Loss: 0.0922, Accuracy: 98.12%
Epoch [2/2], Loss: 0.0892, Accuracy: 98.21%


In [ ]:
test_csv_path = "/kaggle/input/dl-lab4-dataset/isic2020/test.csv"

test_model(model, test_dataloader, criterion, device)

# Saving the Predictons
predictions = []
image_names = []

with torch.no_grad():
    for images, paths in test_dataloader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        
        for img_path, pred in zip(paths, predicted.cpu().numpy()):
            image_name = os.path.basename(img_path).replace(".jpg", "")
            image_names.append(image_name)
            predictions.append("benign" if pred == 0 else "malignant")


test_df = pd.read_csv(test_csv_path)
test_df["target"] = test_df["image_name"].map(dict(zip(image_names, predictions)))
test_df.to_csv("/kaggle/working/test_with_predictions.csv", index=False)

Test Loss: 0.0907, Test Accuracy: 98.17%

Classification Report:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      2142
           1       0.00      0.00      0.00        40

    accuracy                           0.98      2182
   macro avg       0.49      0.50      0.50      2182
weighted avg       0.96      0.98      0.97      2182

AUC ROC Score: 0.5000
F1 Score: 0.9726


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


-----------------

---------------